# Configuration Tables Demo

This notebook shows how to load a Model Zoo configuration that sources
hyperparameters and coefficient priors from CSV tables. It reuses the marketing
mix example so you can see how table-driven edits propagate without touching the
JSON payload.

## Prerequisites

- The repository root must be available so we can reference `configs/`.
- `requests` is only needed if you want to POST the modified config to the API;
  the core of this notebook uses in-memory loaders.
- Optional: run `docker compose up --build` if you want to register the config
  with the FastAPI service and execute a run.

In [ ]:
#!pip install requests pandas --quiet

## Locate Config and Tables

`marketing_mix.json` references `marketing_mix_coefficients.csv` for its
coefficient priors. We load both so you can inspect the raw tables.

In [1]:
import json
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

CONFIG_PATH = (PROJECT_ROOT / "configs" / "marketing_mix.json").resolve()
COEFFS_PATH = (PROJECT_ROOT / "configs" / "marketing_mix_coefficients.csv").resolve()

print(f"Config path: {CONFIG_PATH}")
print(f"Coefficient table: {COEFFS_PATH}")

coeffs_df = pd.read_csv(COEFFS_PATH)
coeffs_df

Config path: /Users/subhagatoadak/Documents/GitHub/model-zoo/configs/marketing_mix.json
Coefficient table: /Users/subhagatoadak/Documents/GitHub/model-zoo/configs/marketing_mix_coefficients.csv


,name,distribution,params.mu,params.sigma,params.shape,params.rate,constraints.sign,constraints.lower,constraints.upper
0,tv,normal,2.0,1.0,NaN,NaN,positive,0.0,NaN
1,radio,half_normal,NaN,1.0,NaN,NaN,nonnegative,NaN,NaN
2,social,gamma,NaN,NaN,2.0,1.0,NaN,NaN,NaN


## Load Configuration with Table Support

`load_model_configuration` automatically merges table rows when `tables` are
specified. The coefficient priors appear under `cfg.coefficients` and will
include both inline and table-sourced definitions.

In [2]:
import json
from pathlib import Path
import pandas as pd
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = (PROJECT_ROOT / "configs" / "marketing_mix.json").resolve()
COEFFS_PATH = (PROJECT_ROOT / "configs" / "marketing_mix_coefficients.csv").resolve()

print(f"Config path: {CONFIG_PATH}")
print(f"Coefficient table: {COEFFS_PATH}")

coeffs_df = pd.read_csv(COEFFS_PATH)
coeffs_df

Config path: /Users/subhagatoadak/Documents/GitHub/model-zoo/configs/marketing_mix.json
Coefficient table: /Users/subhagatoadak/Documents/GitHub/model-zoo/configs/marketing_mix_coefficients.csv


,name,distribution,params.mu,params.sigma,params.shape,params.rate,constraints.sign,constraints.lower,constraints.upper
0,tv,normal,2.0,1.0,NaN,NaN,positive,0.0,NaN
1,radio,half_normal,NaN,1.0,NaN,NaN,nonnegative,NaN,NaN
2,social,gamma,NaN,NaN,2.0,1.0,NaN,NaN,NaN


## Modify Tables Without Editing JSON

Here we build a temporary DataFrame that adds a new `digital` channel and pushes
it into the config via a table override. This leaves the original CSV untouched
by writing to a temp file.

In [3]:
import pandas as pd

clean_table = coeffs_df.copy()
clean_table = clean_table[clean_table['name'] != 'radio']
clean_table = pd.concat([
    clean_table,
    pd.DataFrame([
        {
            'name': 'digital',
            'distribution': 'normal',
            'params.mu': 0.6,
            'params.sigma': 0.3,
            'constraints.sign': 'positive'
        }
    ])
], ignore_index=True)

def _row_to_jsonable(row: dict) -> dict:
    out = {}
    for key, value in row.items():
        if isinstance(value, float) and pd.isna(value):
            out[key] = None
        else:
            out[key] = value
    return out

table_rows = [_row_to_jsonable(row) for row in clean_table.to_dict(orient='records')]
print('Preview of modified table rows:')
table_rows[:3]


Preview of modified table rows:


[{'name': 'tv',
  'distribution': 'normal',
  'params.mu': 2.0,
  'params.sigma': 1.0,
  'params.shape': None,
  'params.rate': None,
  'constraints.sign': 'positive',
  'constraints.lower': 0.0,
  'constraints.upper': None},
 {'name': 'social',
  'distribution': 'gamma',
  'params.mu': None,
  'params.sigma': None,
  'params.shape': 2.0,
  'params.rate': 1.0,
  'constraints.sign': None,
  'constraints.lower': None,
  'constraints.upper': None},
 {'name': 'digital',
  'distribution': 'normal',
  'params.mu': 0.6,
  'params.sigma': 0.3,
  'params.shape': None,
  'params.rate': None,
  'constraints.sign': 'positive',
  'constraints.lower': None,
  'constraints.upper': None}]

## Load a Customised Configuration

Clone the original config, point the `coefficient_priors.tables` entry to the
temporary CSV, and reload to see the new priors take effect.

In [4]:
import json
from copy import deepcopy
from src.model_zoo.utils.config_loader import load_model_configuration_from_dict

config_data = json.loads(CONFIG_PATH.read_text())
config_mut = deepcopy(config_data)
priors_section = config_mut.setdefault('coefficient_priors', {})
priors_section['tables'] = [{
    'content': table_rows
}]

cfg_custom = load_model_configuration_from_dict(
    config_mut,
    env='dev',
    base_path=CONFIG_PATH.parent,
)

cfg_custom.coefficients.to_runner_params()


{'coefficient_priors': {'intercept': {'distribution': 'normal',
   'params': {'mu': 100.0, 'sigma': 20.0},
   'constraints': {'sign': 'positive'}},
  'variables': [{'distribution': 'normal',
    'params': {'mu': 2.0, 'sigma': 1.0},
    'name': 'tv',
    'constraints': {'sign': 'positive', 'lower': 0.0}},
   {'distribution': 'gamma',
    'params': {'shape': 2.0, 'rate': 1.0},
    'name': 'social'},
   {'distribution': 'normal',
    'params': {'mu': 0.6, 'sigma': 0.3},
    'name': 'digital',
    'constraints': {'sign': 'positive'}}]},
 'coefficient_constraints': {'intercept': {'sign': 'positive'},
  'variables': {'tv': {'sign': 'positive', 'lower': 0.0},
   'digital': {'sign': 'positive'}}}}

## Optional: Register and Run via API

If the API is running, register the modified config under a new environment and
submit a run. The run endpoint is asynchronous, so we poll until it completes.

### Adjust Hyperparameters

You can tune sampler settings before registering the configuration. The next
cell updates the number of draws and the sigma scale hyperparameter.

In [5]:
values = config_mut.setdefault('hyperparameters', {}).setdefault('values', [])
model_id = config_mut['model']['id']

def _set_value(param: str, value, env: str | None = 'dev'):
    for entry in values:
        if entry.get('model_id') == model_id and entry.get('param') == param and entry.get('env') == env:
            entry['value'] = value
            return
    values.append({'model_id': model_id, 'param': param, 'value': value, 'env': env})

_set_value('draws', 600)
_set_value('sigma_scale', 3.0)
values

[{'model_id': 'pymc_marketing_mix_v1',
  'param': 'draws',
  'value': 600,
  'env': 'dev'},
 {'model_id': 'pymc_marketing_mix_v1',
  'param': 'tune',
  'value': 400,
  'env': 'dev'},
 {'model_id': 'pymc_marketing_mix_v1',
  'param': 'target_accept',
  'value': 0.9,
  'env': 'dev'},
 {'model_id': 'pymc_marketing_mix_v1',
  'param': 'chains',
  'value': 4,
  'env': 'dev'},
 {'model_id': 'pymc_marketing_mix_v1',
  'param': 'cores',
  'value': 1,
  'env': 'dev'},
 {'model_id': 'pymc_marketing_mix_v1',
  'param': 'compute_convergence_checks',
  'value': 'true',
  'env': 'dev'},
 {'model_id': 'pymc_marketing_mix_v1',
  'param': 'sigma_distribution',
  'value': 'half_normal',
  'env': 'dev'},
 {'model_id': 'pymc_marketing_mix_v1',
  'param': 'sigma_scale',
  'value': 3.0,
  'env': 'dev'},
 {'model_id': 'pymc_marketing_mix_v1',
  'param': 'sigma_shape',
  'value': 2.0,
  'env': 'dev'},
 {'model_id': 'pymc_marketing_mix_v1',
  'param': 'sigma_rate',
  'value': 1.0,
  'env': 'dev'}]

In [6]:
import requests
import time

BASE_URL = "http://localhost:8000"
EXPERIMENT_ENV = "dev-table-demo"

register_payload = {
    'env': EXPERIMENT_ENV,
    'model_id': cfg_custom.model['id'],
    'config': config_mut,
}
reg_resp = requests.post(f"{BASE_URL.rstrip('/')}/registry/configs", json=register_payload, timeout=30)
reg_resp.raise_for_status()
print('Registered config for env', EXPERIMENT_ENV)

run_payload = {
    'model_id': cfg_custom.model['id'],
    'env': EXPERIMENT_ENV,
    'data_sources': {
        'tv_spend': [10, 12, 15, 18, 20],
        'radio_spend': [5, 6, 5, 7, 8],
        'social_spend': [2, 3, 4, 4, 5],
        'marketing_rows': [
            {'tv': 10, 'radio': 5, 'social': 2, 'digital': 3},
            {'tv': 12, 'radio': 6, 'social': 3, 'digital': 4},
            {'tv': 15, 'radio': 5, 'social': 4, 'digital': 4.2},
            {'tv': 18, 'radio': 7, 'social': 4, 'digital': 5},
            {'tv': 20, 'radio': 8, 'social': 5, 'digital': 5.5},
        ],
        'sales': [310, 320, 340, 360, 380],
        'digital_spend': [3, 4, 4.2, 5, 5.5]
    },
    'evaluation': {'metric_ids': ['r2_v1', 'mse_v1'], 'y_true_key': 'y'}
}

submit_resp = requests.post(f"{BASE_URL.rstrip('/')}/runs", json=run_payload, timeout=30)
submit_resp.raise_for_status()
run_ticket = submit_resp.json()
print('Submitted run', run_ticket)

status_url = f"{BASE_URL.rstrip('/')}/runs/{run_ticket['run_id']}"
while True:
    status_resp = requests.get(status_url, timeout=30)
    status_resp.raise_for_status()
    payload = status_resp.json()
    if payload['status'] in {'succeeded', 'failed', 'cancelled'}:
        break
    time.sleep(0.5)

payload


Registered config for env dev-table-demo
Submitted run {'run_id': '2e92cb05-5069-44a6-84c8-315ede51d3e7', 'status': 'queued'}


{'run_id': '2e92cb05-5069-44a6-84c8-315ede51d3e7',
 'status': 'succeeded',
 'submitted_at': '2025-10-16T04:36:46.078014',
 'completed_at': '2025-10-16T04:38:16.320427',
 'result': {'run_id': 'run_20251016_043646_081846',
  'status': 'succeeded',
  'predictions': [305.0804393044018,
   325.1593865864568,
   333.52680745730675,
   361.83932821275573,
   381.59891324460625],
  'diagnostics': {'coefficients_model': {'alpha': 216.35613562810875,
    'dv_sigma': 3.067183239591044,
    'coef[tv]': 4.150248586060997,
    'coef[radio]': 7.67539769846943,
    'coef[social]': 3.4643279110546175,
    'coef[digital]': 0.638724500408908},
   'hyperparameters_model': {'draws': 1000,
    'tune': 1000,
    'target_accept': 0.9,
    'chains': 1}},
  'artifacts': {},
  'eval_train': {'r2_v1': 0.9348429170473197, 'mse_v1': 24.02667433880086},
  'eval_valid': {'r2_v1': 0.0, 'mse_v1': 2.55652356377729},
  'summary': {'Call': {'module': 'model_zoo.examples.pymc_models',
    'callable': 'pymc_linear_nuts',
  

## Clean Up

Remember to delete the temporary CSV if you no longer need it.

### Linear Regression Example

The same table-driven approach works for the simple linear regression example.
The cell below reads `configs/linear_regression.json`, injects coefficient
priors, registers it, and submits a run using synthetic data.

In [9]:
from src.model_zoo.utils.config_loader import load_model_configuration
from examples.register_model_via_api import _generate_linear, _poll_run

lin_cfg = load_model_configuration(PROJECT_ROOT / 'configs' / 'linear_regression.json', env='dev')
lin_config = lin_cfg.raw
lin_priors = lin_config.setdefault('coefficient_priors', {})
lin_priors['tables'] = [{
    'content': [
        {
            'name': 'scaled_X',
            'distribution': 'normal',
            'params': {'mu': 2.0, 'sigma': 0.8},
            'constraints': {'sign': 'positive'}
        }
    ]
}]
lin_env = 'dev-linear-demo'
reg_payload = {'env': lin_env, 'model_id': lin_config['model']['id'], 'config': lin_config}
requests.post(f"{BASE_URL.rstrip('/')}/registry/configs", json=reg_payload, timeout=30).raise_for_status()

x_vals, y_vals = _generate_linear(points=50, slope=2.1, intercept=1.0, noise_sigma=0.1)
lin_run_payload = {
    'model_id': lin_config['model']['id'],
    'env': lin_env,
    'data_sources': {'x_values': x_vals, 'y_values': y_vals},
    'evaluation': {'metric_ids': ['r2_v1', 'mse_v1'], 'y_true_key': 'y'}
}
lin_response = requests.post(f"{BASE_URL.rstrip('/')}/runs", json=lin_run_payload, timeout=30)
lin_response.raise_for_status()
run_ticket = lin_response.json()
print('Submitted run', run_ticket)

status_url = f"{BASE_URL.rstrip('/')}/runs/{run_ticket['run_id']}"
while True:
    status_resp = requests.get(status_url, timeout=30)
    status_resp.raise_for_status()
    payload = status_resp.json()
    if payload['status'] in {'succeeded', 'failed', 'cancelled'}:
        break
    time.sleep(0.5)
payload


Submitted run {'run_id': 'dc6cb3f9-8c9f-4c89-8b65-362560702a20', 'status': 'queued'}


{'run_id': 'dc6cb3f9-8c9f-4c89-8b65-362560702a20',
 'status': 'succeeded',
 'submitted_at': '2025-10-16T04:41:29.712755',
 'completed_at': '2025-10-16T04:41:29.769575',
 'result': {'run_id': 'run_20251016_044129_715782',
  'status': 'succeeded',
  'predictions': [1.2099079915425683,
   1.4199207686144062,
   1.6299335456862423,
   1.8399463227580792,
   2.049959099829917,
   2.259971876901754,
   2.469984653973591,
   2.6799974310454284,
   2.890010208117265,
   3.1000229851891024,
   3.3100357622609398,
   3.5200485393327767,
   3.730061316404614,
   3.9400740934764507,
   4.150086870548288,
   4.3600996476201255,
   4.570112424691962,
   4.780125201763799,
   4.990137978835636,
   5.200150755907473,
   5.410163532979311,
   5.620176310051148,
   5.830189087122985,
   6.040201864194822,
   6.250214641266659,
   6.460227418338497,
   6.670240195410334,
   6.88025297248217,
   7.090265749554008,
   7.300278526625845,
   7.510291303697683,
   7.72030408076952,
   7.9303168578413565,
   8